# Re-calibrar spain-france con PnLCalib — solo las 3 ventanas etiquetadas

Corre pnlcalib SOLO en los frames de los 3 bloques etiquetados (min 10-18, 26-34,
64-72) y guarda la homografia imagen->cancha(cm) por frame. NO re-trackea: la
re-proyeccion del tracking existente se hace despues, local.

**Prerrequisito:** subir el video completo a
`MyDrive/football_analytics/videos/spain-france.mp4` (1,4 GB).

## 1) Repo + PnLCalib + Drive

In [ ]:
import os
REPO_DIR='/content/ncf_event_tracker'; BRANCH='events-model'
if not os.path.exists(REPO_DIR):
    !git clone -q --branch {BRANCH} https://github.com/pipachiesa/ncf_event_tracker.git {REPO_DIR}
%cd {REPO_DIR}
!git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!pip install -q -U ultralytics supervision huggingface_hub pyyaml
# PnLCalib
if not os.path.exists('/content/PnLCalib'):
    !git clone -q https://github.com/mguti97/PnLCalib.git /content/PnLCalib
    !mkdir -p /content/PnLCalib/weights
    for w in ('SV_FT_WC14_kp','SV_FT_WC14_lines'):
        !wget -q -O /content/PnLCalib/weights/{w} https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/{w}
os.environ['PNLCALIB_DIR']='/content/PnLCalib'
from google.colab import drive; drive.mount('/content/drive')
print('setup OK, weights:', os.listdir('/content/PnLCalib/weights'))

## 2) RUTAS + ventanas etiquetadas (se leen solas de los groundtruth)

In [ ]:
import csv
DRIVE_DIR='/content/drive/MyDrive/football_analytics'
VIDEO_PATH=os.path.join(DRIVE_DIR,'videos','spain-france.mp4')
assert os.path.exists(VIDEO_PATH), f'FALTA el video: {VIDEO_PATH}  (subilo a Drive)'
STRIDE=2          # frame_stride del tracking del partido (meta: frame_stride 2)
EVERY=5           # calcular homografia cada 5 frames CSV (como main.py)
PAD=60            # margen de frames alrededor de cada bloque
GT=[os.path.join(REPO_DIR,'events_model','dataset',f) for f in
    ('spain-france_10m_18m_proposed_groundtruth.csv',
     'spain-france_26m_34m_V2_groundtruth.csv',
     'spain-france_64m_72m_proposed_groundtruth.csv')]
wins=[]
for g in GT:
    fr=[int(float(r['Start Frame'])) for r in csv.DictReader(open(g)) if r.get('Start Frame')]
    wins.append((max(1,min(fr)-PAD), max(fr)+PAD))
frames=set()
for lo,hi in wins:
    frames.update(range(lo, hi+1, EVERY))
frames=sorted(frames)
print('ventanas (frame CSV):', wins)
print('frames a calibrar:', len(frames))
OUT=os.path.join(DRIVE_DIR,'spain-france_homographies.json')

## 3) Correr PnLCalib (GPU). Cachea a Drive: si se corta, re-corré y sigue.

In [ ]:
import cv2, json, numpy as np, torch, time
import sys; sys.path.insert(0, os.path.join(REPO_DIR,'data_cleanup'))
from pitch_calib import PnLCalibrator
dev='cuda' if torch.cuda.is_available() else 'cpu'
cal=PnLCalibrator(device=dev); print('pnlcalib en', dev)
H={}
if os.path.exists(OUT):
    H={int(k):v for k,v in json.load(open(OUT)).items()}
    print('reanudando, ya hay', len(H))
cap=cv2.VideoCapture(VIDEO_PATH)
t0=time.time()
for n,k in enumerate(frames):
    if k in H: continue
    cap.set(cv2.CAP_PROP_POS_FRAMES, (k-1)*STRIDE)
    ok,frame=cap.read()
    if not ok: H[k]=None; continue
    m=cal.homography(frame)
    H[k]= (m.flatten().tolist() if m is not None else None)
    if n%100==0:
        json.dump(H, open(OUT,'w'))
        done=sum(1 for v in H.values() if v); el=time.time()-t0
        print(f'{n}/{len(frames)}  ok={done}  {el:.0f}s')
cap.release(); json.dump(H, open(OUT,'w'))
ok=sum(1 for v in H.values() if v)
print(f'LISTO: {ok}/{len(frames)} frames calibrados -> {OUT}')

## 4) Bajar el archivo de homografias

In [ ]:
from google.colab import files
files.download(OUT)
print('bajalo y ponelo en la Mac; yo sigo local con la re-proyeccion + re-Viterbi + AUC.')